# 02 — Pipeline de limpieza

**Objetivo:** documentar cada transformación aplicada al CSV crudo y verificar que el resultado es correcto antes de pasarlo al EDA.

**Conclusiones:**
- El CSV usa `;` como separador y contiene 1,251 filas × 9 columnas.
- Dos columnas de porcentaje (`Answer Rate`, `Service Level`) estaban como strings → convertidas a float [0, 1].
- Tres columnas de tiempo en formato `HH:MM:SS` → convertidas a segundos enteros.
- La columna `Index` es un row number redundante → eliminada.
- El CSV procesado se guarda en `data/processed/call_center_clean.csv`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Asegura que src/ esté en el path cuando se ejecuta desde notebooks/
sys.path.insert(0, str(Path.cwd().parent / "src"))

from call_center.data_loader import load_raw, RAW_COLS, CLEAN_COLS
from call_center.cleaning import parse_percentage, parse_hhmmss_to_seconds, clean, save_processed

## 1. Carga del CSV crudo

In [ ]:
raw = load_raw()
print(f"Shape: {raw.shape}")
raw.head()

In [ ]:
# Tipos originales — aquí se ve el problema
raw.dtypes

In [ ]:
# Nulos por columna
raw.isnull().sum()

## 2. Transformación de porcentajes

`"94.01%"` → `0.9401` (escala [0, 1])

In [ ]:
# Antes
sample = raw[[RAW_COLS["answer_rate"], RAW_COLS["service_level"]]].head()
print("Antes:")
print(sample)
print(sample.dtypes)

# Después
print("\nDespués:")
print(parse_percentage(raw[RAW_COLS["answer_rate"]]).head())

## 3. Transformación de tiempos HH:MM:SS → segundos

In [ ]:
time_cols = [RAW_COLS["answer_speed"], RAW_COLS["talk_duration"], RAW_COLS["waiting_time"]]

print("Antes:")
print(raw[time_cols].head())

print("\nDespués (segundos):")
for col in time_cols:
    print(f"  {col}: {parse_hhmmss_to_seconds(raw[col]).head().tolist()}")

## 4. Pipeline completo

In [ ]:
df = clean(raw)
print(f"Shape: {df.shape}")
df.dtypes

In [ ]:
df.head()

## 5. Estadísticas descriptivas del dataset limpio

In [ ]:
df.describe().round(3)

## 6. Verificación de invariantes del negocio

Antes de guardar, validamos que los datos tienen sentido operacionalmente.

In [ ]:
checks = {
    "Sin nulos": df.isnull().sum().sum() == 0,
    "answer_rate en [0,1]": df[CLEAN_COLS["answer_rate"]].between(0, 1).all(),
    "service_level en [0,1]": df[CLEAN_COLS["service_level"]].between(0, 1).all(),
    "answered <= incoming": (df[CLEAN_COLS["answered"]] <= df[CLEAN_COLS["incoming"]]).all(),
    "tiempos no negativos": (
        (df[CLEAN_COLS["answer_speed"]] >= 0) &
        (df[CLEAN_COLS["talk_duration"]] >= 0) &
        (df[CLEAN_COLS["waiting_time"]] >= 0)
    ).all(),
    "1251 filas": len(df) == 1251,
}

for check, result in checks.items():
    status = "OK" if result else "FALLO"
    print(f"  [{status}] {check}")

## 7. Guardar CSV procesado

In [ ]:
out = save_processed(df)
print(f"Guardado en: {out}")